# Project Orientation

This curriculum builds a toy autoregressive language model from raw tensors up to a compact GPT-style implementation. The goal is not to compete with production LLM stacks; it is to make every important equation, shape, and systems tradeoff inspectable.


## Why Start From Scratch?

High-level libraries are excellent once the invariants are already clear. Early on, though, abstractions can hide the exact object we are trying to model: a conditional distribution over the next token in a sequence.

In this project we first expose the math directly, then bridge to reusable modules. That order matters because later topics such as sparse attention, quantization, JEPA-style objectives, and world models all modify one of the same basic ingredients: representation, objective, memory, or information flow.


## Problem: What Must This Curriculum Compute?

At the start of the project, the key object is not yet a neural network but a workflow. We need a reproducible way to choose a device, select a runtime profile, and know which notebooks progressively turn symbols into a working language model.

We will first inspect those choices directly so every configuration value is visible. Only after the numerical checks pass will we rely on the later model code.

Why this matters later: when an architecture paper claims to improve attention, memory, quantization, or world modeling, it is usually changing one of these constraints: what information can move across positions, how much memory is required, what objective is optimized, or which representations are preserved.


## Runtime Profiles And Device Order

The project exposes three runtime profiles:

- `quick`: small shapes and short runs for smoke tests and fast iteration.
- `study`: enough capacity to observe the main learning dynamics without long waits.
- `stretch`: larger settings that make the scaling pressures more visible.

Device selection follows `CUDA -> MPS -> CPU`. That order prefers the most capable accelerator available while keeping the notebooks portable on a laptop.


In [ ]:
from llm_from_scratch.configs import get_device, profile_config

device = get_device()
quick_model, quick_train = profile_config("quick")
device, quick_model, quick_train


In [ ]:
quick_model, quick_train = profile_config("quick")
study_model, study_train = profile_config("study")
stretch_model, stretch_train = profile_config("stretch")

assert device.type in {"cuda", "mps", "cpu"}
assert quick_model.block_size < study_model.block_size < stretch_model.block_size
assert quick_model.n_embd < study_model.n_embd < stretch_model.n_embd
assert quick_train.max_steps < study_train.max_steps < stretch_train.max_steps

{
    "device": device.type,
    "quick_block_size": quick_model.block_size,
    "study_block_size": study_model.block_size,
    "stretch_block_size": stretch_model.block_size,
}


## Notebook Sequence

The first five notebooks establish the first-principles path:

1. orientation and runtime profiles
2. tensors, autograd, and probability
3. tokenization from characters toward subwords
4. embeddings, logits, and next-token loss
5. attention from explicit score matrices

After that, the curriculum wraps those primitives into transformer blocks, training loops, generation, finetuning, quantization, and modern architecture comparisons.


In [ ]:
profiles = {
    "quick": profile_config("quick"),
    "study": profile_config("study"),
    "stretch": profile_config("stretch"),
}

summary = {
    name: {
        "block_size": model.block_size,
        "n_embd": model.n_embd,
        "n_head": model.n_head,
        "head_dim": model.n_embd // model.n_head,
        "attention_scores_per_head": model.block_size ** 2,
        "max_steps": train.max_steps,
    }
    for name, (model, train) in profiles.items()
}

assert summary["quick"]["attention_scores_per_head"] < summary["study"]["attention_scores_per_head"] < summary["stretch"]["attention_scores_per_head"]
summary


## Verification Commands And Long-Term Bridges

The main verification commands for this curriculum are:

- `uv run python scripts/smoke_notebooks.py --quick`
- `uv run pytest -q`
- `uv run python scripts/strip_notebook_outputs.py notebooks`

Those checks keep the learning material runnable as the project grows. They also matter for later topics: sparse attention attacks the `T^2` score matrix, quantization reduces model and KV-cache memory, JEPA-style training changes the objective, and world models broaden what the representation is expected to predict.


In [ ]:
from pathlib import Path

expected_notebooks = [
    "00_project_orientation.ipynb",
    "01_tensors_autograd_and_probability.ipynb",
    "02_text_tokenization_char_to_subword.ipynb",
    "03_embeddings_and_language_modeling.ipynb",
    "04_attention_from_raw_tensors.ipynb",
]
notebook_dir = Path(".")
script_dir = Path("../scripts")

found = sorted(path.name for path in notebook_dir.glob("*.ipynb"))
assert all((script_dir / name).exists() for name in ["smoke_notebooks.py", "strip_notebook_outputs.py"])
assert all(name in found for name in expected_notebooks)
assert len(found) >= 13

{"notebook_count": len(found), "first_five": found[:5]}


## Exercise Checkpoint 1

1. Why is `quick` the right default when you are checking notebook execution rather than model quality?
2. If `n_embd = 128` and `n_head = 4`, what is the head dimension, and why must that ratio stay integral?


## Exercise Checkpoint 2

1. Suppose a future notebook increases sequence length from `64` to `2048`. Which quantity in this curriculum tells you attention will become much more expensive?
2. Name one long-term topic in the bridge above and state which core constraint it changes: objective, memory, representation, or information flow.
